### Estimating Dipole Moment of SO+

Using the pyscf and pyberny packages. Code generated by Gemini Pro 3, edited and commented by BAM.

In [1]:
import pyscf
from pyscf import dft
from pyscf.geomopt.berny_solver import optimize 
from pyscf.hessian import thermo

In [2]:
#Define the SO+ molecule
mol = pyscf.M(
    atom='S 0.0 0.0 0.0; O 0.0 0.0 1.424',
    #basis='6-311++g(d,p)',
    basis='aug-cc-pVTZ',
    spin=1,
    charge=1,
    unit='Angstrom'
)

In [3]:
#Build the Unrestricted DFT object (UKS) and set the functional
mf = dft.UKS(mol)
mf.xc = 'm06-2x'

In [4]:
#Perform the Geometry Optimization
print("--- Starting Geometry Optimization (M06-2X / aug-cc-pVTZ) ---")
mol_eq = optimize(mf)

--- Starting Geometry Optimization (M06-2X / aug-cc-pVTZ) ---

Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   S   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   O   0.000000   0.000000   1.424000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr

converged SCF energy = -472.97482372568  <S^2> = 0.7542849  2S+1 = 2.0042803
--------------- UKS_Scanner gradients ---------------
         x                y                z
0 S    -0.0000000000     0.0000000000    -0.0121856556
1 O     0.0000000000     0.0000000000     0.0122531806
----------------------------------------------
cycle 1: E = -472.974823726  dE = -472.975  norm(grad) = 0.0172809

Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   S   0.000000   0.000000   0.005961    0.000000  0.000000  0.005961
   O   0.000000   0.0000

In [5]:
print("\nOptimized Geometry Structure:")
print(mol_eq.tostring())


Optimized Geometry Structure:
S           0.00000000        0.00000000        0.00390653
O           0.00000000        0.00000000        1.42009347


In [6]:
#Run a final single-point calculation on the optimized structure.
# 'optimize()' returns a new Mole object (mol_eq) with updated coordinates.
print("\n--- Running Final Calculation on Optimized Geometry ---")
mf_eq = dft.UKS(mol_eq)
mf_eq.xc = 'm06-2x'
mf_eq.kernel()


--- Running Final Calculation on Optimized Geometry ---
converged SCF energy = -472.974915190165  <S^2> = 0.75422871  2S+1 = 2.0042243


-472.9749151901648

In [7]:
#Extract and print the final dipole moment
print("\n--- Final Dipole Moment Results ---")
dipole_vector = mf_eq.dip_moment()


--- Final Dipole Moment Results ---

WARN: System has nonzero charge 1; the dipole moment is origin-dependent.
Location of origin: [0. 0. 0.]

Dipole moment(X, Y, Z, Debye):  0.00000,  0.00000, -0.10529


In [8]:
#Compute Rotational Constants for comparison to lab work (B = 23249.1 from Amano:1991:519)
print("\n=== ROTATIONAL CONSTANT ===")
# PySCF stores coordinates in Bohr internally; thermo.rotation_const expects Bohr and AMU
masses = mol_eq.atom_mass_list()
coords = mol_eq.atom_coords()

# Calculate the constants in MHz
rot_constants = thermo.rotation_const(masses, coords, unit='GHz')*1000.
print(f'A: {rot_constants[0]:.2f} MHz\nB: {rot_constants[1]:.2f} MHz\nC: {rot_constants[2]:.2f} MHz')


=== ROTATIONAL CONSTANT ===
A: inf MHz
B: 23623.67 MHz
C: 23623.67 MHz


Given that these are B_e values, this is close enough to the Amano work for the dipole to be a reasonable estimate.